<a href="https://colab.research.google.com/github/aounraza379/research_paper_intelligence_assistant/blob/main/research_paper_intelligence_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install groq -q

In [4]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

In [5]:
# Testing API work
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in 6 words."}]
)

print(response.choices[0].message.content)

Hello, it is very nice.


In [6]:
!pip install pymupdf -q # handles text extraction more reliably and is faster

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 86.7 MB/s eta 0:00:00


In [7]:
from google.colab import files
uploaded = files.upload()  # upload PDF

Saving SpatialTrust_Research.docx to SpatialTrust_Research.docx


In [8]:
#  Text Extracter
import fitz  # PyMuPDF

filename = list(uploaded.keys())[0]  # uploaded file
print(filename) # our file name
doc = fitz.open(filename)

full_text = ""
for page_num, page in enumerate(doc):
    page_text = page.get_text()
    full_text += page_text
    print(f"--- Page {page_num + 1} preview ---")
    print(page_text[:500])  # first 500 chars of each page for inspection
    print()

print(f"\nTotal characters extracted: {len(full_text)}")

SpatialTrust_Research.docx
--- Page 1 preview ---
Spatial-Trust: A Multi-Factor Reputation Framework for Mitigating
Sybil-Attack Misinformation in Peer-to-Peer Disaster Response
Systems
Aoun Raza
Department of Computer Science
Abstract — Peer-to-peer (P2P) platforms have emerged as critical
infrastructure for decentralized disaster response, enabling affected
populations to report resource needs and coordinate relief without
dependence on centralized authority. However, the open
participation model that makes these platforms valuable also
rende

--- Page 2 preview ---
substantially reducing the likelihood — under the modeled feature
distributions — that genuine distress signals are suppressed.
Second, all posts originating beyond 150 km satisfy S(P) ≤ 0.20 <
θ, rendering distant-origin Sybil posts consistently detectable within
the defined zone geometry. Third, any adversary with GPS-spoofing
capability achieves S(P) ≥ 0.45 > θ across all post ages,
establishing an upper bound on detec

In [9]:
# Document Chunking
def chunk_text(text, chunk_size=1000, overlap=200):
    """
    Splits text into overlapping chunks.
    chunk_size: max characters per chunk
    overlap: characters repeated between consecutive chunks
    """
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # overlapping

    return chunks

chunks = chunk_text(full_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print("\n--- First chunk ---")
print(chunks[0])
print("\n--- Second chunk (notice overlap with end of first) ---")
print(chunks[1])

Total chunks created: 83

--- First chunk ---
Spatial-Trust: A Multi-Factor Reputation Framework for Mitigating
Sybil-Attack Misinformation in Peer-to-Peer Disaster Response
Systems
Aoun Raza
Department of Computer Science
Abstract — Peer-to-peer (P2P) platforms have emerged as critical
infrastructure for decentralized disaster response, enabling affected
populations to report resource needs and coordinate relief without
dependence on centralized authority. However, the open
participation model that makes these platforms valuable also
renders them vulnerable to Sybil attacks, wherein adversaries
deploy multiple fabricated identities to inject misinformation, divert
rescue resources, and degrade coordination integrity under
conditions of maximum societal stress.
Existing countermeasures are fundamentally ill-suited to the
disaster context: reputation systems require longitudinal user
histories unavailable among first-time survivors; supervised learning
approaches demand pre-labeled trai

In [9]:
# Install Sentence Transformers
!pip install sentence-transformers -q

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed just the first chunk to see what an embedding actually looks like
sample_embedding = model.encode(chunks[0])

print(f"Embedding shape: {sample_embedding.shape}")
print(f"First 10 values: {sample_embedding[:10]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (384,)
First 10 values: [ 0.01484743 -0.05920223  0.03275495  0.02041184  0.08240414  0.0115169
  0.02899917 -0.01039908  0.01779586  0.06221531]


In [15]:
# Testing semantic similarity works!
from sklearn.metrics.pairwise import cosine_similarity

test_sentences = [
    "Sybil attacks use fake identities to manipulate systems.",
    "Adversaries deploy multiple fabricated accounts to inject misinformation.",  # similar meaning, different words
    "The weather in Karachi is hot in summer."  # unrelated
]

test_embeddings = model.encode(test_sentences)

sim_1_2 = cosine_similarity([test_embeddings[0]], [test_embeddings[1]])[0][0]
sim_1_3 = cosine_similarity([test_embeddings[0]], [test_embeddings[2]])[0][0]

print(f"Similarity (Sybil sentence vs. paraphrased Sybil sentence): {sim_1_2:.3f}")
print(f"Similarity (Sybil sentence vs. unrelated weather sentence): {sim_1_3:.3f}")

Similarity (Sybil sentence vs. paraphrased Sybil sentence): 0.504
Similarity (Sybil sentence vs. unrelated weather sentence): -0.068


In [17]:
# Now embed all chunks
chunk_embeddings = model.encode(chunks, show_progress_bar=True)

print(f"Total embeddings created: {len(chunk_embeddings)}")
print(f"Shape of embeddings array: {chunk_embeddings.shape}")

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Total embeddings created: 83
Shape of embeddings array: (83, 384)
